# CardioIA — classificação textual de risco com TF-IDF

Notebook da entrega obrigatória da Fase 2. A base é simulada e o resultado é exclusivamente acadêmico: não representa diagnóstico ou triagem clínica real.

## 1. Objetivo e rastreabilidade

O experimento transforma frases curtas em vetores TF-IDF e treina uma Regressão Logística para distinguir os rótulos simulados **baixo risco** e **alto risco**.

As versões feminina e masculina de um mesmo cenário compartilham o mesmo `id_cenario`. A divisão é feita por esse identificador para impedir que versões quase equivalentes apareçam simultaneamente em treino e teste.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

RAIZ = Path.cwd()
if not (RAIZ / "fase2").exists():
    RAIZ = Path.cwd().parents[1]

sys.path.insert(0, str(RAIZ / "fase2" / "src"))

from classificar_risco_texto import (
    analisar_vies_contrafactual,
    avaliar_modelo,
    carregar_dataset,
    criar_pipeline,
    dividir_por_cenario,
    termos_mais_influentes,
)

## 2. Dataset simulado

O arquivo possui frases rotuladas e pares contrafactuais por sexo. Os rótulos foram definidos apenas para demonstrar o pipeline solicitado no enunciado.

In [ ]:
dados = carregar_dataset()
resumo_dataset = pd.DataFrame({
    "total_frases": [len(dados)],
    "cenarios_unicos": [dados["id_cenario"].nunique()],
    "duplicadas": [int(dados["frase"].duplicated().sum())],
})
display(resumo_dataset)
display(pd.crosstab(dados["situacao"], dados["sexo_referencia"]))

## 3. Separação sem vazamento por cenário

A divisão reserva 20% dos cenários de cada classe para teste. As duas versões demográficas permanecem sempre no mesmo conjunto.

In [ ]:
treino, teste = dividir_por_cenario(dados)
assert not set(treino["id_cenario"]).intersection(teste["id_cenario"])

display(pd.DataFrame({
    "conjunto": ["treino", "teste"],
    "frases": [len(treino), len(teste)],
    "cenarios": [treino["id_cenario"].nunique(), teste["id_cenario"].nunique()],
}))
display(pd.crosstab(teste["situacao"], teste["sexo_referencia"]))

## 4. TF-IDF e Regressão Logística

O TF-IDF e o classificador ficam no mesmo `Pipeline`. Assim, o vocabulário e os pesos são ajustados apenas com as frases de treino.

In [ ]:
modelo = criar_pipeline()
modelo.fit(treino["frase"], treino["situacao"])

vetorizador = modelo.named_steps["tfidf"]
pd.DataFrame({
    "configuracao": ["Vocabulário", "N-gramas", "Classes"],
    "valor": [
        len(vetorizador.vocabulary_),
        str(vetorizador.ngram_range),
        ", ".join(modelo.classes_),
    ],
})

## 5. Avaliação no conjunto de teste

Além da acurácia solicitada, são calculados precisão, recall, F1 e ROC AUC. Em triagem, observar apenas a acurácia pode esconder erros importantes na classe de maior risco.

In [ ]:
metricas, previsoes = avaliar_modelo(modelo, teste)
metricas_resumo = {
    chave: valor
    for chave, valor in metricas.items()
    if chave != "relatorio_classificacao"
}
display(pd.DataFrame([metricas_resumo]).style.format({
    "acuracia": "{:.3f}",
    "precisao_alto_risco": "{:.3f}",
    "recall_alto_risco": "{:.3f}",
    "f1_alto_risco": "{:.3f}",
    "roc_auc": "{:.3f}",
}))
display(pd.DataFrame(metricas["relatorio_classificacao"]).T)

In [ ]:
matriz = confusion_matrix(
    teste["situacao"],
    previsoes["predicao"],
    labels=["baixo risco", "alto risco"],
)
ConfusionMatrixDisplay(
    matriz,
    display_labels=["Baixo risco", "Alto risco"],
).plot(cmap="RdPu", colorbar=False)
plt.title("Matriz de confusão — conjunto de teste")
plt.show()

## 6. Termos associados às classes

Os coeficientes descrevem associações aprendidas nesta base simulada. Eles não demonstram causalidade ou relevância médica universal.

In [ ]:
termos = termos_mais_influentes(modelo, quantidade=12)
display(termos.groupby("direcao", group_keys=False).head(12))

## 7. Análise de viés contrafactual

Comparamos versões feminina e masculina do mesmo relato. Como sintomas e rótulo são mantidos, uma grande variação indicaria sensibilidade indevida ao marcador textual de sexo.

Essa análise complementa — e não substitui — a avaliação tabular por sexo já existente no projeto.

In [ ]:
resumo_vies, pares = analisar_vies_contrafactual(modelo, teste)
display(pd.DataFrame([resumo_vies]).style.format("{:.4f}"))
display(pares.sort_values("diferenca_absoluta", ascending=False).head(10))

## 8. Conclusões e limitações

- O pipeline exigido de TF-IDF e classificação textual foi implementado e avaliado em dados não vistos.
- A separação por cenário impede vazamento entre pares contrafactuais.
- O teste por sexo verifica estabilidade textual em exemplos equivalentes.
- A base é pequena, balanceada e totalmente simulada; seu balanceamento não representa prevalência.
- Os dados textuais, tabulares e visuais têm origens distintas e permanecem como modalidades independentes.
- O modelo não possui validação clínica, externa ou prospectiva e não deve ser usado em atendimentos reais.